# ⚡ CIC-IDS-2018: Smart Sampling Pipeline (Minority Hunter)


In [1]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
import gc
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import LabelEncoder, StandardScaler, Normalizer

warnings.filterwarnings('ignore')



## 1. Smart Sampling (The Minority Hunter)
Scans each file in chunks (preventing OOM errors on 4GB files). Extracts *all* attack rows (preserving rare attacks natively). If attacks < 15,000 per file, fills remainder with randomly sampled Benign traffic.


In [2]:
DATA_DIR = r"c:\Users\PMLS\Desktop\Cases_FYP\CIC-IDS-2018"
all_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
df_parts = []
TOTAL_TARGET_PER_FILE = 15000

print("--- Stage: Smart Sampling (The Minority Hunter) ---")
print(f"Discovered {len(all_files)} CSV files.\n")

for file in all_files:
    print(f"Scanning -> {os.path.basename(file)}")
    attack_rows = []
    benign_rows = []
    
    # Process in chunks to handle potential 4GB+ files safely out-of-core
    for chunk in pd.read_csv(file, chunksize=500_000, low_memory=False, encoding='utf-8'):
        chunk.columns = chunk.columns.str.strip()
        
        # Determine actual label column safely
        label_col = 'Label'
        if 'Label' not in chunk.columns:
            matches = [c for c in chunk.columns if 'label' in c.lower()]
            if matches: label_col = matches[0]
            
        if label_col in chunk.columns:
            # Drop repeated headers (fixes the string repeated headers issue)
            chunk = chunk[chunk[label_col] != 'Label']
            
            # Separate Attacks vs Benign (case-insensitive for BENIGN)
            is_benign = chunk[label_col].astype(str).str.upper() == 'BENIGN'
            attack_rows.append(chunk[~is_benign])
            benign_rows.append(chunk[is_benign])
            
    # Combine chunks
    df_attack = pd.concat(attack_rows, ignore_index=True) if attack_rows else pd.DataFrame()
    df_benign = pd.concat(benign_rows, ignore_index=True) if benign_rows else pd.DataFrame()
    
    n_attacks = len(df_attack)
    n_benign  = len(df_benign)
    print(f"   [Inventory] Attacks: {n_attacks:,} | Benign: {n_benign:,}")
    
    # Minority Hunter Logistics: Keep ALL attacks. Fill with benign if needed up to 15,000.
    if n_attacks < TOTAL_TARGET_PER_FILE:
        diff = TOTAL_TARGET_PER_FILE - n_attacks
        if n_benign > 0:
            sample_size = min(diff, n_benign)
            df_benign_sampled = df_benign.sample(n=sample_size, random_state=42)
            final_df = pd.concat([df_attack, df_benign_sampled], ignore_index=True)
        else:
            final_df = df_attack
    else:
        # We KEEP ALL attacks unconditionally
        final_df = df_attack
        
    print(f"   [Selected] Shape for this file: {final_df.shape}")
    
    # Standardize label column name
    if label_col != 'Label' and label_col in final_df.columns:
        final_df.rename(columns={label_col: 'Label'}, inplace=True)
        
    df_parts.append(final_df)
    gc.collect()

# Combine all generated files
df = pd.concat(df_parts, ignore_index=True)
del df_parts
gc.collect()

print(f"\nInitial Combined Shape: {df.shape}")



--- Stage: Smart Sampling (The Minority Hunter) ---
Discovered 10 CSV files.

Scanning -> 02-14-2018.csv
   [Inventory] Attacks: 380,949 | Benign: 667,626
   [Selected] Shape for this file: (380949, 80)
Scanning -> 02-15-2018.csv
   [Inventory] Attacks: 52,498 | Benign: 996,077
   [Selected] Shape for this file: (52498, 80)
Scanning -> 02-16-2018.csv
   [Inventory] Attacks: 601,802 | Benign: 446,772
   [Selected] Shape for this file: (601802, 80)
Scanning -> 02-20-2018.csv
   [Inventory] Attacks: 576,191 | Benign: 7,372,557
   [Selected] Shape for this file: (576191, 84)
Scanning -> 02-21-2018.csv
   [Inventory] Attacks: 687,742 | Benign: 360,833
   [Selected] Shape for this file: (687742, 80)
Scanning -> 02-22-2018.csv
   [Inventory] Attacks: 362 | Benign: 1,048,213
   [Selected] Shape for this file: (15000, 80)
Scanning -> 02-23-2018.csv
   [Inventory] Attacks: 566 | Benign: 1,048,009
   [Selected] Shape for this file: (15000, 80)
Scanning -> 02-28-2018.csv
   [Inventory] Attacks: 68

## 2. Full Attack Inventory
Ensures rare attacks (Heartbleed, SQL Injection, etc.) are present.


In [3]:
print("Full Attack Inventory (Unique Labels found across all files):")
if 'Label' in df.columns:
    label_counts = df['Label'].value_counts()
    for lbl, count in label_counts.items():
        print(f" - {lbl}: {count:,}")
else:
    print("WARNING: Label column not found in combined dataframe.")



Full Attack Inventory (Unique Labels found across all files):
 - DDOS attack-HOIC: 686,012
 - DDoS attacks-LOIC-HTTP: 576,191
 - DoS attacks-Hulk: 461,912
 - Bot: 286,191
 - FTP-BruteForce: 193,360
 - SSH-Bruteforce: 187,589
 - Infilteration: 161,934
 - DoS attacks-SlowHTTPTest: 139,890
 - DoS attacks-GoldenEye: 41,508
 - Benign: 29,072
 - DoS attacks-Slowloris: 10,990
 - DDOS attack-LOIC-UDP: 1,730
 - Brute Force -Web: 611
 - Brute Force -XSS: 230
 - SQL Injection: 87


## 3. Data Type Correction & NaN Pre-Cleaning
Convert types and filter broken Label dependencies.


In [4]:
print(f"\n--- Stage: Cleanup ---")
print(f"Current df.shape: {df.shape}")

# Drop rows missing labels
shape_before = df.shape
if 'Label' in df.columns:
    df.dropna(subset=['Label'], inplace=True)
print(f"Shape before dropping rows with missing Labels: {shape_before}")
print(f"Shape AFTER dropping rows with missing Labels: {df.shape}")

# Convert all features to numeric & force infinite to NaN
cols = [c for c in df.columns if c != 'Label']
for c in cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df.replace([np.inf, -np.inf], np.nan, inplace=True)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Count of numeric vs non-numeric columns: {len(numeric_cols)} numeric vs {len(object_cols)} object columns")

X_raw = df.drop(columns=['Label'])
y_raw = df['Label']
print(f"Current df.shape (post-cleanup): {df.shape}")




--- Stage: Cleanup ---
Current df.shape: (2777307, 84)
Shape before dropping rows with missing Labels: (2777307, 84)
Shape AFTER dropping rows with missing Labels: (2777307, 84)
Count of numeric vs non-numeric columns: 83 numeric vs 1 object columns
Current df.shape (post-cleanup): (2777307, 84)


## 4. Imputation
SimpleImputer with Median Strategy.


In [5]:
print(f"\n--- Stage: Imputation ---")
print(f"Current df.shape structure: {df.shape}")

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_raw)

print(f"Imputation Complete!")




--- Stage: Imputation ---
Current df.shape structure: (2777307, 84)
Imputation Complete!


## 5. Variance Thresholding (Low 0.01 Threshold)
Bypasses exact threshold if remaining feature map count goes below 5.


In [6]:
print(f"\n--- Stage: Thresholding ---")
print(f"Current structural shape input: {X_imputed.shape}")

selector = VarianceThreshold(threshold=0.01)

# Use block to guard against extreme drops
X_sel_temp = selector.fit_transform(X_imputed)

cols_before = X_raw.shape[1]
cols_after = X_sel_temp.shape[1]

print(f"Features BEFORE threshold: {cols_before}")

if cols_after < 5:
    print(f"Features AFTER threshold: {cols_after} -> (< 5 features). Skipping threshold and keeping all numeric columns!")
    X_selected = X_imputed
    final_feature_names = X_raw.columns.tolist()
else:
    print(f"Features AFTER threshold: {cols_after}")
    X_selected = X_sel_temp
    final_feature_names = [X_raw.columns[i] for i in selector.get_support(indices=True)]




--- Stage: Thresholding ---
Current structural shape input: (2777307, 79)
Features BEFORE threshold: 83
Features AFTER threshold: 66


## 6. Scaling & Normalization
LabelEncoder -> StandardScaler -> Normalizer (L2)


In [7]:
print(f"\n--- Stage: Final Transformations ---")
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

normalizer = Normalizer(norm='l2')
X_normalized = normalizer.fit_transform(X_scaled)

print("Scaling & Normalization Complete.")




--- Stage: Final Transformations ---
Scaling & Normalization Complete.


## 7. Artifact Export
Generates CIC_IDS_2018_Preprocessed_Combined_2.csv.


In [8]:
df_final = pd.DataFrame(X_normalized, columns=final_feature_names)
df_final['Label'] = y_encoded

output_file = "CIC_IDS_2018_Preprocessed_Combined_2.csv"
print(f"\nExporting to: {output_file}...")
df_final.to_csv(output_file, index=False)
print(f"✅ Success! Final Exported shape: {df_final.shape}")




Exporting to: CIC_IDS_2018_Preprocessed_Combined_2.csv...
✅ Success! Final Exported shape: (2777307, 67)
